# Chandler/Modelski historical urban population -- wrangle to long format

Source: `data/chandler-modelski/` (Reba, Reitsma & Seto 2016 supplementary data --
Chandler + Modelski Ancient + Modelski Modern). Wide format (one column per attested
year) reconciled here following the authors' own `scientifcDataCombineDatasets.r`.

In [1]:
# Cell 1
%matplotlib inline
import pandas as pd
from pathlib import Path
from scripts.shared import db_utils

ROOT = Path(db_utils.__file__).parent.parent.parent
RAW  = ROOT / 'data' / 'chandler-modelski'

# Source files are Latin-1, not UTF-8 (confirmed byte-level -- e.g. "A Coru\xf1a").
# low_memory=False avoids a DtypeWarning on chandlerV2.csv's mixed-type year columns
# (some cells hold a stray whitespace-only string instead of being truly empty).
chandler = pd.read_csv(RAW / 'chandlerV2.csv', encoding='latin-1', low_memory=False)
modelski_ancient = pd.read_csv(RAW / 'modelskiAncientV2.csv', encoding='latin-1', low_memory=False)
modelski_modern  = pd.read_csv(RAW / 'modelskiModernV2.csv', encoding='latin-1', low_memory=False)

# One row (Ecbatana, modelski_ancient) has a trailing space on Latitude ("34.8... "),
# which pushes that whole column to object dtype -- coerce Lat/Lon to numeric everywhere
# defensively rather than patch that one row.
for df in (chandler, modelski_ancient, modelski_modern):
    df['Latitude']  = pd.to_numeric(df['Latitude'].astype(str).str.strip())
    df['Longitude'] = pd.to_numeric(df['Longitude'].astype(str).str.strip())

print(f"chandler:         {chandler.shape}")
print(f"modelski_ancient: {modelski_ancient.shape}")
print(f"modelski_modern:  {modelski_modern.shape}")
print(chandler.columns[:8].tolist())

chandler:         (1597, 812)
modelski_ancient: (154, 47)
modelski_modern:  (293, 7)
['City', 'OtherName', 'Country', 'Latitude', 'Longitude', 'Certainty', 'BC_2250', 'BC_2000']


In [2]:
# Cell 2
ID_COLS = ['City', 'OtherName', 'Country', 'Latitude', 'Longitude', 'Certainty']

def melt_wide(df):
    year_cols = [c for c in df.columns if c not in ID_COLS]
    long = df.melt(id_vars=ID_COLS, value_vars=year_cols, var_name='year_label', value_name='pop')
    # errors='coerce' also catches non-empty-but-non-numeric cells (e.g. a stray ' ')
    long['pop'] = pd.to_numeric(long['pop'], errors='coerce')
    long = long.dropna(subset=['pop']).copy()
    long['year'] = (long['year_label']
                     .str.replace('BC_', '-', regex=False)
                     .str.replace('AD_', '', regex=False)
                     .astype(int))
    return long.drop(columns='year_label')

modern_long = melt_wide(modelski_modern)
print(modern_long.shape)
print(modern_long.head().to_string())

(293, 8)
          City  OtherName        Country   Latitude   Longitude  Certainty      pop  year
0      Abidjan     Abijan  Cote d'Ivoire   5.309660   -4.012660          1  1929000  2000
1        Adana        NaN         Turkey  37.001670   35.328890          1  1041000  2000
2  Addis Ababa        NaN       Ethiopia   9.024970   38.746890          1  2424000  2000
3     Adelaide        NaN      Australia -34.928660  138.598630          1  1092000  2000
4    Ahmadabad  Ahmedabad          India  23.039568   72.566004          1  2954000  2000


In [3]:
# Cell 3
ancient_long = melt_wide(modelski_ancient)
chandler_long = melt_wide(chandler)

print(f"modelski_ancient long: {ancient_long.shape}, years {ancient_long['year'].min()}..{ancient_long['year'].max()}")
print(f"chandler long:         {chandler_long.shape}, years {chandler_long['year'].min()}..{chandler_long['year'].max()}")
print(chandler_long.sort_values('year').head().to_string())

modelski_ancient long: (621, 8), years -3700..1000
chandler long:         (9438, 8), years -2250..1975
         City OtherName Country   Latitude  Longitude  Certainty      pop  year
412      Ebla       NaN   Syria  35.798398  36.798728          1  30000.0 -2250
1010   Nippur       NaN    Iraq  32.126939  45.230785          1  30000.0 -2250
2507  Memphis       NaN   Egypt  29.844722  31.250833          2  60000.0 -2000
2950     Susa       NaN    Iran  32.190850  48.246028          1  25000.0 -2000
3072       Ur       NaN    Iraq  30.962659  46.103210          1  65000.0 -2000


In [4]:
# Cell 4
def add_cityid(df, source):
    df = df.copy()
    df['cityid'] = (df['City'].astype(str) + '_' +
                     df['Latitude'].round(1).astype(str) + '_' +
                     df['Longitude'].round(1).astype(str))
    df['source'] = source
    return df

combined = pd.concat([
    add_cityid(chandler_long, 'chandler'),
    add_cityid(ancient_long, 'modelski_ancient'),
    add_cityid(modern_long, 'modelski_modern'),
], ignore_index=True)
print(f"combined: {combined.shape}")

exact_dupe_mask = combined.duplicated(subset=['cityid', 'year', 'pop'], keep=False)
print(f"rows sharing cityid+year+pop with another row: {exact_dupe_mask.sum()}")

conflict_mask = combined.duplicated(subset=['cityid', 'year'], keep=False) & ~exact_dupe_mask
print(f"rows sharing cityid+year with a DIFFERING pop: {conflict_mask.sum()}")
print(combined[conflict_mask].sort_values(['cityid', 'year']).to_string())

combined: (10352, 10)
rows sharing cityid+year+pop with another row: 44
rows sharing cityid+year with a DIFFERING pop: 130
               City                         OtherName       Country   Latitude   Longitude  Certainty        pop  year                  cityid            source
81       Alexandria                               NaN         Egypt  31.200092   29.918739          1   300000.0  -200    Alexandria_31.2_29.9          chandler
9805     Alexandria                               NaN         Egypt  31.200092   29.918739          1   600000.0  -200    Alexandria_31.2_29.9  modelski_ancient
158      Alexandria                               NaN         Egypt  31.200092   29.918739          1   100000.0   500    Alexandria_31.2_29.9          chandler
9929     Alexandria                               NaN         Egypt  31.200092   29.918739          1   200000.0   500    Alexandria_31.2_29.9  modelski_ancient
271      Alexandria                               NaN         Egypt  31.

In [5]:
# Cell 5
# Drop true redundancy (same cityid+year+pop attested by both sources) -- keep one.
# Leave genuine disagreements (same cityid+year, differing pop) as separate rows,
# each still carrying its own `source` -- these are competing attestations, not an
# error to resolve by picking a winner (per LPF's approach to contested temporal
# data: keep multiple timespans/estimates with provenance rather than collapse them).
final = combined.drop_duplicates(subset=['cityid', 'year', 'pop'], keep='first').reset_index(drop=True)
print(f"final: {final.shape}  (combined was {combined.shape}, dropped {len(combined) - len(final)})")

n_conflict_pairs = final.duplicated(subset=['cityid', 'year'], keep=False).sum()
print(f"rows still involved in a cityid+year conflict (kept, not resolved): {n_conflict_pairs}")

print(final[final['City'] == 'Ur'].sort_values('year').to_string())

final: (10330, 10)  (combined was (10352, 10), dropped 22)
rows still involved in a cityid+year conflict (kept, not resolved): 130
     City OtherName Country   Latitude  Longitude  Certainty       pop  year        cityid            source
9462   Ur       NaN    Iraq  30.962659   46.10321          1   12000.0 -2800  Ur_31.0_46.1  modelski_ancient
9484   Ur       NaN    Iraq  30.962659   46.10321          1   10000.0 -2500  Ur_31.0_46.1  modelski_ancient
9508   Ur       NaN    Iraq  30.962659   46.10321          1   10000.0 -2400  Ur_31.0_46.1  modelski_ancient
9532   Ur       NaN    Iraq  30.962659   46.10321          1   20000.0 -2300  Ur_31.0_46.1  modelski_ancient
9551   Ur       NaN    Iraq  30.962659   46.10321          1   40000.0 -2200  Ur_31.0_46.1  modelski_ancient
9567   Ur       NaN    Iraq  30.962659   46.10321          1  100000.0 -2100  Ur_31.0_46.1  modelski_ancient
4      Ur       NaN    Iraq  30.962659   46.10321          1   65000.0 -2000  Ur_31.0_46.1          chandl

In [6]:
# Cell 6
out = final.rename(columns={
    'City': 'city', 'OtherName': 'other_name', 'Country': 'country',
    'Latitude': 'lat', 'Longitude': 'lon', 'Certainty': 'certainty',
})[['cityid', 'city', 'other_name', 'country', 'lat', 'lon', 'certainty', 'year', 'pop', 'source']]
out = out.sort_values(['cityid', 'year']).reset_index(drop=True)

OUT_DIR = ROOT / 'output' / 'edop' / 'kgreview'
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / 'chandler_modelski_long.csv'
out.to_csv(out_path, index=False)
print(f"wrote {len(out)} rows to {out_path}")
print(out.head().to_string())

wrote 10330 rows to /Users/karlg/Documents/repos/_edops/output/edop/kgreview/chandler_modelski_long.csv
               cityid      city        other_name  country        lat      lon  certainty  year      pop    source
0  A Coruna_43.4_-8.4  A Coruna  Coruna, A Coruña    Spain  43.362344 -8.41154          1  1900  43000.0  chandler
1     Aachen_50.8_6.1    Aachen   Aix-la-Chapelle  Germany  50.776640  6.08342          1  1300  21000.0  chandler
2     Aachen_50.8_6.1    Aachen   Aix-la-Chapelle  Germany  50.776640  6.08342          1  1500  18000.0  chandler
3     Aachen_50.8_6.1    Aachen   Aix-la-Chapelle  Germany  50.776640  6.08342          1  1600  16000.0  chandler
4     Aachen_50.8_6.1    Aachen   Aix-la-Chapelle  Germany  50.776640  6.08342          1  1700  15000.0  chandler
